# Notebook 03 — Job Analyzer
**Mission Veteran AI · Veteran Career Navigator · **

This notebook tests `analyze_job_description` — pipeline step 5 of the full pipeline.
It loads PT-001 from `knowledge-bases/prompt_templates.json`, calls IBM Granite via
watsonx.ai, and returns structured job analysis with alignment scoring.

**Prerequisites**
- `ibm-watsonx-ai` installed  
- `.env` file with `WATSONX_API_KEY`, `WATSONX_URL`, `WATSONX_PROJECT_ID`  
- `knowledge-bases/prompt_templates.json` present with PT-001

In [ ]:
%pip install -q ibm-watsonx-ai python-dotenv

In [ ]:
import json
import os
import sys
from dotenv import load_dotenv

load_dotenv()

# Add the project root to sys.path so we can import tool helpers
ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

API_KEY    = os.environ.get('WATSONX_API_KEY', '')
WX_URL     = os.environ.get('WATSONX_URL', 'https://us-south.ml.cloud.ibm.com')
PROJECT_ID = os.environ.get('WATSONX_PROJECT_ID', '')

assert API_KEY,    'Set WATSONX_API_KEY in your .env'
assert PROJECT_ID, 'Set WATSONX_PROJECT_ID in your .env'
print('Credentials loaded ✓')

## 1 · Load PT-001 from tblPrompts

In [ ]:
KB_PATH = os.path.join(ROOT, 'knowledge-bases', 'prompt_templates.json')

with open(KB_PATH, 'r', encoding='utf-8') as f:
    templates = json.load(f)

template = next(t for t in templates if t['name'] == 'job_description_analysis')
print(f"Loaded: {template['template_id']} — {template['description']}")
print(f"Model params: {template['model_params']}")

## 2 · Define test job description

In [ ]:
JOB_DESCRIPTION = """
Supply Chain Operations Manager
Mid-level · Logistics / Distribution

We are seeking a Supply Chain Operations Manager to oversee end-to-end
logistics operations across three regional distribution centers.

Required:
- 3-5 years of supply chain or logistics management experience
- Proven ability to manage cross-functional teams of 10 or more
- Inventory management and demand forecasting
- Vendor and supplier relationship management
- Budget oversight and cost reduction initiatives
- Proficiency with ERP or WMS systems

Preferred:
- APICS CSCP or CPIM certification
- Six Sigma Green Belt or higher
- Experience with SAP or Oracle SCM

Keywords: supply chain, logistics, inventory, KPI, cross-functional,
procurement, vendor management, warehouse operations, cost reduction
"""

# Sample veteran skill inventory (would come from map_military_skills)
VETERAN_SKILLS = [
    {"skill_name": "logistics coordination",        "category": "Operations",      "confidence": "High"},
    {"skill_name": "team leadership",               "category": "Leadership",      "confidence": "High"},
    {"skill_name": "inventory management",          "category": "Supply Chain",    "confidence": "High"},
    {"skill_name": "budget oversight",              "category": "Finance",         "confidence": "Medium"},
    {"skill_name": "cross-functional coordination", "category": "Collaboration",   "confidence": "High"},
]
print('Inputs defined ✓')

## 3 · Call IBM Granite via watsonx.ai

In [ ]:
import re
from ibm_watsonx_ai import APIClient, Credentials
from ibm_watsonx_ai.foundation_models import ModelInference
from ibm_watsonx_ai.metanames import GenTextParamsMetaNames as Params

MODEL_ID = 'ibm/granite-4-h-small'

credentials = Credentials(url=WX_URL, api_key=API_KEY)
client      = APIClient(credentials=credentials)
params      = template['model_params']

model = ModelInference(
    model_id=MODEL_ID,
    api_client=client,
    project_id=PROJECT_ID,
    params={
        Params.MAX_NEW_TOKENS:    params['max_new_tokens'],
        Params.MIN_NEW_TOKENS:    params['min_new_tokens'],
        Params.TEMPERATURE:       params['temperature'],
        Params.REPETITION_PENALTY:params['repetition_penalty'],
    },
)

system_prompt   = template['system_prompt']
user_template_s = template['user_prompt_template']
user_prompt     = user_template_s.format(job_description=JOB_DESCRIPTION.strip()[:2000])

response = model.generate(
    prompt=f'System:\n{system_prompt}\nUser:\n{user_prompt}\nAssistant:\n',
)

raw = response['results'][0]['generated_text'].strip()
raw = re.sub(r'^```[a-z]*\n?', '', raw)
raw = re.sub(r'\n?```$', '', raw)

print('Raw model output:')
print(raw)

## 4 · Parse and display structured analysis

In [ ]:
analysis = json.loads(raw)

print(f"Role:       {analysis.get('role_title', '—')}")
print(f"Seniority:  {analysis.get('seniority_level', '—')}")
print(f"Industry:   {analysis.get('industry', '—')}")
print()
print('Required skills:')
for s in analysis.get('required_skills', []):
    print(f'  • {s}')
print()
print('Preferred skills:')
for s in analysis.get('preferred_skills', []):
    print(f'  • {s}')
print()
print('ATS keywords:', ', '.join(analysis.get('ats_keywords', [])))

## 5 · Compute alignment score against veteran skill inventory

In [ ]:
def normalise(text):
    return re.sub(r'[^a-z0-9 ]', '', text.lower()).strip()

required = analysis.get('required_skills', [])
matched, unmatched = [], []

for req in required:
    req_n = normalise(req)
    hit = any(
        req_n in normalise(vs['skill_name']) or normalise(vs['skill_name']) in req_n
        for vs in VETERAN_SKILLS
    )
    (matched if hit else unmatched).append(req)

pct = round(len(matched) / len(required) * 100) if required else 0

print(f'Alignment: {pct}% ({len(matched)}/{len(required)} required skills covered)')
print()
print('Matched:')
for s in matched:   print(f'  ✓ {s}')
print('Gaps:')
for s in unmatched: print(f'  ✗ {s}')

## 6 · Test the tool function directly

In [ ]:
# Load the tool function directly from the toolkit for testing
import sys, os
ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

from toolkits.plan_builder.build_plan import analyze_job_description

result = analyze_job_description(
    job_description=JOB_DESCRIPTION,
)

print(json.dumps(result, indent=2))
